# core

> the vault: one SQLite file holding everything you have read, and the retrieval over it

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A `Vault` is a [litesearch](https://github.com/Karthik777/litesearch) `Index`: docs to nodes to chunks, FTS plus vectors, RRF. Retrieval defaults come from there.

The vault adds `kind` on every document, shelves in one file, an entity graph, and the acquisition, extraction and code verbs. Open a path, or nothing for `~/.vishalakshi/vault.db`.


In [ ]:
#| export
import json, os, re, time, uuid, warnings
from collections import Counter
import numpy as np
from functools import wraps
from inspect import Parameter, signature
from fastcore.all import AttrDict, L, Path, first, ifnone, patch, store_attr
from litesearch import (Index, DTYPE, dir2files, hash_embed, static_embedder, topic_nodes,
                        DOC_EXTS, write_txn)

In [ ]:
#| export
KINDS = ('web', 'pdf', 'arxiv', 'youtube', 'file', 'code', 'data', 'note', 'image')
_window, DFLT_ENC = re.compile(r'^Pages \d+(?:–\d+)?:'), 'minishlab/potion-multilingual-128M'

def tidy_bc(bc:str) -> str:
    "Drop `build_tree`'s `Pages n–m:` window placeholders from a breadcrumb: real nodes, noise in a citation."
    return ' › '.join(dict.fromkeys(p for p in map(str.strip, (bc or '').split('›')) if p and not _window.match(p)))

def kinds(kind) -> L:
    "A kind filter as a list: `'note'`, `'note,web'` and `['note','web']` all work."
    return L(kind.split(',') if isinstance(kind, str) else kind).filter()

#: Alias to model id. Three: prose, source, and the ONNX one for when faithfulness beats speed.
#: Any other model2vec id works by name, and `encoder=` takes an object with an `.encode`.
ENCODERS = {'multilingual': DFLT_ENC,                                    # 100+ languages, Devanagari included
            'code':         'minishlab/potion-code-16M-v2',              # identifiers; what kosha embeds with
            'gemma':        'onnx-community/embeddinggemma-300m-ONNX'}   # ~300M, the most faithful and the slowest


In [ ]:
#| export
class HashEmbed:
    "litesearch's `hash_embed` behind an `.encode`, so an offline vault is an encoder like any other."
    def __init__(self, dims:int=256, dtype=DTYPE): store_attr()
    def encode(self, xs, **kw): return hash_embed(list(xs), ndim=self.dims, dtype=self.dtype)

def _load(nm:str, dtype):
    'The encoder for a model id, and what to call the method. ONNX ids need onnxruntime, so it is imported here.'
    if nm != ENCODERS['gemma']: return static_embedder(nm), 'model2vec'
    from litesearch import FastEncode, embedding_gemma
    return FastEncode(embedding_gemma, dtype=dtype), 'onnx'

def mk_encoder(model=None,          # an `ENCODERS` alias, any model2vec id, or an object with `.encode`
               dims:int=256,        # dims for the hashing fallback only
               offline:bool=False,  # skip the download attempt entirely
               dtype=DTYPE,         # stored width; litesearch's default everywhere
) -> AttrDict:
    'The best encoder available as `AttrDict(model, dims, method, name, note)`.'
    if offline:
        return AttrDict(model=HashEmbed(dims, dtype), dims=dims, method='hash', name='hash',
        note=f'char-n-gram hashing ({dims}d): lexical only; pass encoder= or restore network access for real semantics')
    # a str has `.encode` too, so the id path has to be excluded before duck-typing
    if not isinstance(model, (str, bytes)) and hasattr(model, 'encode'):
        v = model.encode(['probe']); nm = type(model).__name__
        return AttrDict(model=model, dims=int(v.shape[-1]), method='given', name=nm,
                        note=f'{nm} ({v.shape[-1]}d, {np.dtype(dtype)}, yours)')
    nm = ENCODERS.get(model, model or DFLT_ENC)
    try:
        m, meth = _load(nm, dtype)
        v = m.encode(['probe'])
        return AttrDict(model=m, dims=int(v.shape[-1]), method=meth, name=nm,
                        note=f'{nm} ({v.shape[-1]}d, {np.dtype(dtype)}, {meth})')
    except Exception as e:
        warnings.warn(f'could not load {nm} ({type(e).__name__}: {str(e)[:100]}); falling back to hash_embed')
        return mk_encoder(dims=dims, offline=True, dtype=dtype)

In [ ]:
#| hide
# mk_encoder returns the encoder; `Index` is what turns it into doc and query functions
_h = mk_encoder(offline=True)
test_eq(_h.model.encode(['a', 'b']).shape, (2, 256))
test_eq(_h.model.encode(['a']).dtype, np.float16)
test_eq((_h.method, _h.dims, _h.name), ('hash', 256, 'hash'))

class _Fwd:                       # a model2vec-shaped embedder: `doc_encoder` drops kw for these
    def encode(self, xs): return np.zeros((len(xs), 8), dtype=np.float16)
# `given`, not `model2vec`: an embedder you built is not a model2vec model
test_eq((mk_encoder(_Fwd()).dims, mk_encoder(_Fwd()).method), (8, 'given'))

In [ ]:
#| export
class Vault(Index):
    '''Everything you have read, in one SQLite file, searchable as one corpus.'''
    _any_noisy = None
    def __init__(self,
                 path:str=None,       # vault file; None -> ~/.vishalakshi/vault.db
                 encoder=None,        # an `ENCODERS` alias, a model2vec id, an `mk_encoder()` result, or None
                 store:str='store',   # chunk store name
                 offline:bool=None,   # never attempt a model download; None -> $VISHALAKSHI_OFFLINE
                 dims:int=256,        # dims for the hashing fallback
                 db=None):            # an open litesearch Database to share; shelves pass the vault's
        offline = bool(os.getenv('VISHALAKSHI_OFFLINE')) if offline is None else offline
        self.enc = encoder if isinstance(encoder, AttrDict) and 'dims' in encoder \
                   else mk_encoder(encoder, dims=dims, offline=offline)
        super().__init__(ifnone(path, Path.home()/'.vishalakshi'/'vault.db'), encoder=self.enc.model, name=store, db=db)
        self._register()

    def _where(self, kind=None, include_noisy:bool=False) -> str|None:
        '''A chunk-store `WHERE` for kind and quality policy, pushed into retrieval.'''
        clauses = []
        if kinds(kind): clauses.append(_kw(kind))
        sub = None if include_noisy else self._noisy_sql()
        if not (clauses or sub): return None
        docq = f'doc_id IN (SELECT id FROM [{self.t.prefix}docs] WHERE {" AND ".join(clauses)})' if clauses else None
        return ' AND '.join(x for x in (docq, sub) if x)

    def __repr__(self):
        s = self.stats()
        return (f"Vault({self.path!r}: {s['docs']} docs, {s['chunks']} chunks, {s['entities']} entities, encoder={self.enc.method})")

def _kw(kind) -> str: return 'kind IN (%s)' % ','.join(map(repr, kinds(kind)))


In [ ]:
#| export
@patch
def add(self:Vault,
        src,                  # text, `[(page_no, text)]`, a file path, or a directory
        title:str=None,       # document title; defaults to the filename, or the first line of text
        source:str=None,      # url or path; defaults to the title. Identity is hashed over it
        kind:str=None,        # one of KINDS: the facet you filter and report on
        meta:dict=None,       # provenance: the query that found it, when, which tier fetched it
        force:bool=False,     # re-ingest a source already present
        **kw                  # forwarded to litesearch add_doc (chunker, summarize, with_heading)
) -> dict:
    '''Ingest anything into the vault: tree, chunks, embeddings, ANN index.'''
    # a document's text is not a path, and asking the filesystem about a 40kB "filename" raises
    p = Path(src) if isinstance(src, (str, Path)) and len(str(src)) < 255 and '\n' not in str(src) else None
    if p is not None and p.is_dir():  return self.add_dir(str(p), kind=kind, **kw)
    if p is not None and p.is_file(): return self.add_file(str(p), title=title, kind=kind, **kw)
    ttl = title or _first_line(src)
    return self.db.add_doc(src, ttl, source=source, kind=kind or 'file', store=self.name,
                           emb_fn=self.emb, meta=meta, force=force, **kw)

def _first_line(src, n:int=80) -> str:
    'A title for text that came without one: the first non-empty line, as `toc()` will show it.'
    txt = src if isinstance(src, str) else '\n'.join(t for _, t in (src or []))
    return (first(l.strip().lstrip('# ') for l in txt.splitlines() if l.strip()) or 'untitled')[:n]

@patch
def assets(self:Vault, name:str=None) -> Path:
    'Where extracted assets (PDF images, downloaded papers) go: beside the vault file. litesearch picks it.'
    return self.db.assets(name)

@patch
def add_file(self:Vault, path:str, title:str=None, kind:str=None, **kw) -> dict:
    'Ingest one local file into the vault: tree, chunks, embeddings, ANN index.'
    return self.db.add_file(path, title=title, kind=kind, store=self.name, emb_fn=self.emb, **kw)

@patch
def add_files(self:Vault,
              files,                # paths to ingest, all onto *this* shelf
              kind:str=None,        # override the kind inferred from the extension
              n_workers:int=None,   # parse workers; 0 is serial, None picks by parse-heavy file count
              embed_batch:int=2000, # chunks embedded and written per flush; 0 writes per document
              **kw                  # forwarded to add_file
) -> L:
    '''Ingest a list of files onto this shelf, batched the way litesearch batches a whole tree.'''
    files = L(files).map(Path)
    # an empty list would still reach `rebuild_index`, which is a real cost for no documents
    if not files: return L()
    return L(self.db.add_dir(files=files, store=self.name, kind=kind, emb_fn=self.emb,
                             n_workers=n_workers, embed_batch=embed_batch, **kw))

@patch
def add_dir(self:Vault, dir:str, types:str=DOC_EXTS, kind:str=None,
            route:bool=True,      # send Sanskrit sources to the Sanskrit shelf, file by file
            **kw) -> L:
    """Ingest every document under a directory. `dir2files` skips dotfiles, tests, build and dist."""
    fs = dir2files(dir, types=types)
    if not route or self.name != 'store': return self.add_files(fs, kind=kind, **kw)
    sa, rest = L(), L()
    for f in fs: (sa if is_sanskrit_file(f) else rest).append(f)
    out = self.add_files(rest, kind=kind, **kw)
    if sa: out += self.route('sanskrit').add_files(sa, kind=kind, **kw)
    return out

@patch
def note(self:Vault,
         text:str,            # what you want to remember
         title:str=None,      # defaults to the first line
         tags:list=None,      # free-form tags, kept in the doc's meta
) -> dict:
    'Write a note into the vault so it is searched alongside the corpus.'
    ttl = title or (text.strip().splitlines() or ['note'])[0].lstrip('# ')[:80]
    return self.add(text.strip(), ttl, source=f'note:{uuid.uuid4().hex[:12]}', kind='note', meta=dict(tags=list(tags or [])))


`context` returns whole sections for an LLM. Each section carries a breadcrumb and a `node_id` for
`read`. `related` uses the entity graph. `code=` appends kosha hits when a code database exists.


In [ ]:
#| export
#: `pii=` and `pii_ner=`, spliced onto every gated method so the CLI, the MCP tools and the docs
#: all see them. `@wraps` alone would report the undecorated signature and hide both.
_PII_PARAMS = [Parameter('pii', Parameter.KEYWORD_ONLY, default='off', annotation=str),
               Parameter('pii_ner', Parameter.KEYWORD_ONLY, default=False, annotation=bool)]

def gate(f):
    'Apply the `pii=` policy to what a retrieval method returns, and put its two params on the signature.'
    @wraps(f)
    def _f(self, *a, pii:str='off', pii_ner:bool=False, **kw):
        o = f(self, *a, **kw)
        if pii == 'off' or o is None: return o
        from vishalakshi.pii import gated
        # `read` names the shelf a section came from; for the rest `gated` falls back to this one
        return gated(o, pii, self, ner=pii_ner, store=kw.get('store') or self.name)
    ps = list(signature(f).parameters.values())
    at = next((i for i, p in enumerate(ps) if p.kind is Parameter.VAR_KEYWORD), len(ps))
    _f.__signature__ = signature(f).replace(parameters=ps[:at] + _PII_PARAMS + ps[at:])
    return _f

@patch
@gate
def search(self:Vault,
           q:str,              # query
           limit:int=10,       # hits to return
           kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
           chars:int=300,      # chars of each hit kept as `snippet`
           rerank:bool=False,  # reorder the candidates with a cross-encoder (see below)
           include_noisy:bool=False, # include documents explicitly marked as noisy
           **kw                # forwarded to litesearch doc_search
) -> L:
    """Chunk-level hybrid search; hits carry breadcrumb and `node_id`. Honours kind and noisy marks."""
    hits = Index.search(self, q, limit=limit, rerank=rerank,
                        where=self._where(kind, include_noisy), **kw)
    return L(AttrDict(node_id=h.get('node_id'), doc_id=h.get('doc_id'), page=h.get('page'),
                      breadcrumb=tidy_bc(h.get('breadcrumb')), score=h.get('_rrf_score'),
                      snippet=(h.get('content') or '')[:chars]) for h in hits)

@patch
@gate
def sections(self:Vault, q:str, limit:int=5, kind:str=None, per:int=3, rerank:bool=False,
             include_noisy:bool=False, **kw) -> list:
    'Ranked *sections* rather than chunks. Noisy documents are excluded unless requested.'
    secs = Index.sections(self, q, limit=limit, per=per,
                          where=self._where(kind, include_noisy), rerank=rerank, **kw)
    for s in secs: s['breadcrumb'] = tidy_bc(s.get('breadcrumb'))
    return secs

@patch
@gate
def context(self:Vault,
            q:str,              # the question
            sections:int=6,     # operative sections returned
            related:int=8,      # related sections reached by graph + vector
            kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
            max_read:int=6000,  # chars of assembled text per section
            code:int=None,      # code sections to append; None -> 4 if kosha has indexed the repo
            shelves:int=2,      # sections to append from each *other* shelf; 0 -> none
            dir:str=None,       # repo for the code legs; None -> the cwd repo
            rerank:bool=False,  # reorder the chunk hits before they are rolled up into sections
            include_noisy:bool=False, # include documents explicitly marked as noisy
            **kw                # forwarded to litesearch context
) -> AttrDict:
    'The retrieval an LLM should be handed: whole sections plus what they connect to. sections carry `text, breadcrumb, pages, filename` and their tree neighbourhood;'
    # litesearch already fans out sections*3; don't multiply again
    ctx = Index.context(self, q, related=related, max_read=max_read, sections=sections,
                        where=self._where(kind, include_noisy), rerank=rerank, **kw)
    for r in (*ctx.results, *ctx.related): r.breadcrumb = tidy_bc(r.breadcrumb)
    # retune this shelf before federated legs; code hits are not scored here
    ctx = self._post(q, ctx)
    ctx.encoder, ctx.code, ctx.shelves = self.enc.note, 0, 0
    if shelves:
        found = self.elsewhere(q, limit=shelves)
        ctx.results, ctx.shelves = ctx.results + found, len(found)
    if code or code is None:
        from vishalakshi.code import code_sections, kosha_indexed
        if kosha_indexed(dir):
            hits = code_sections(self, q, n=code or 4, dir=dir)
            ctx.results, ctx.code = ctx.results + hits, len(hits)
    return ctx

@patch
def related(self:Vault, node_id:str, limit:int=8, clip=300) -> L:
    'Sections nearest an existing one .Reuses the vectors usearch already holds, so nothing is re-embedded.'
    out = {}
    for r in self.store(where=f'node_id={node_id!r}', select='rowid as rowid'):
        for n in self.store.ann_neighbors(r['rowid'], limit=limit*3, dtype=DTYPE, columns=['content', 'node_id', 'doc_id']):
            nid = n.get('node_id')
            if nid and nid != node_id and nid not in out:
                out[nid] = dict(node_id=nid, doc_id=n.get('doc_id'), dist=n.get('_dist'),
                                breadcrumb=tidy_bc(self.db.breadcrumb(nid, self.name)), snippet=(n.get('content') or '')[:clip])
            if len(out) >= limit: return L(out.values())
    return L(out.values())

@patch
@gate
def read(self:Vault, node_id:str, max_chars:int=6000,
         store:str=None,      # the shelf the section is on; None -> this one
         ) -> dict:
    'Assemble a whole section back out of its chunks.`store` opens a section on another shelf, so a `node_id` from `elsewhere()` can be read without opening that shelf.'
    return Index.read(self, node_id, store=store or self.name, max_chars=max_chars)

A `node_id` names a section and a `doc_id` names a document. `document` / `read` / `toc` / `sections` walk that tree. Meta lives on the document; a re-ingest rewrites it, so durable judgments go in `doc_marks` instead.

In [ ]:
#| export
@patch
def doc(self:Vault, ref:str) -> dict:
    'One document row, by `doc_id`, exact `source`, or a title substring; `meta` already decoded.'
    # empty ref must not become title LIKE '%%' (would mark the newest doc)
    if not str(ref or '').strip(): return None
    q = str(ref or '').replace("'", "''")
    for w in (f"id='{q}'", f"source='{q}'", f"title LIKE '%{q}%'"):
        if r:=first(self.t.docs(where=w, order_by='added_at desc')): return dict(r, meta=json.loads(r['meta'] or '{}'))
    return None

@patch
@gate
def document(self:Vault,
             ref:str,               # doc_id, source (url or path), or a title substring
             max_chars:int=40000,   # cap on the text returned
             headings:bool=True,    # put the node titles back as markdown headings
             disk:bool=True,        # fall back to a path on disk the vault has never seen
) -> AttrDict:
    'One whole document, reassembled in document order: the unit a model reads to extract from.'
    d = self.doc(ref)
    if d is None:
        p = Path(ref or '')
        if not (disk and p.is_file()): raise ValueError(f'no document in the vault matching {ref!r}')
        txt = p.read_text(errors='replace')
        return AttrDict(doc_id=None, title=p.name, source=str(p), kind='file', meta={}, pages=None,
                        origin='disk', nodes=0, chars=len(txt), truncated=len(txt) > max_chars,
                        text=txt[:max_chars])
    did = d['id'].replace("'", "''")
    chunks = {}
    for c in self.store(where=f"doc_id='{did}'", select='content, node_id, page, rowid as rowid'):
        chunks.setdefault(c['node_id'], []).append(c)
    nodes, parts = sorted(self.t.nodes(where=f"doc_id='{did}'"), key=lambda r: r['seq']), []
    for nd in nodes:
        if headings and nd['level'] and (t := (nd['title'] or '').strip()) and not _window.match(t):
            parts.append('#'*min(nd['level'], 6) + ' ' + t)
        parts += [c['content'] for c in sorted(chunks.get(nd['id'], []), key=lambda c: (c['page'] or 0, c['rowid']))]
    txt = '\n\n'.join(p for p in parts if (p or '').strip())
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], kind=d['kind'],
                    meta=d['meta'], pages=d['pages'], origin='vault', nodes=len(nodes),
                    chars=len(txt), truncated=len(txt) > max_chars, text=txt[:max_chars])

@patch
def set_meta(self:Vault, doc_id:str, **kv) -> dict:
    "Merge key/values into one document's `meta`, and return the merged dict."
    did = (doc_id or '').replace("'", "''")
    r = first(self.t.docs(where=f"id='{did}'"))
    if not r: raise ValueError(f'no document {doc_id!r} in the vault')
    m = {**json.loads(r['meta'] or '{}'), **kv}
    self.t.docs.update(dict(id=doc_id, meta=json.dumps(m, default=str)))
    return m

One vault file, several shelves. A shelf is a partition: its own chunk store, tree and ANN index, so two corpora stop diluting each other's ranking. `shelf(name)` opens one; `drop_shelf` removes it. Acquisition routes some kinds via `KIND_SHELF`.


In [ ]:
#| export
@patch
def _stores(self:Vault):
    'Shelf encoder registry.'
    return self.db.t.vault_stores

@patch
def _rankers(self:Vault):
    'Ranker configuration by shelf.'
    return self.db.t.rankers

@patch
def _ensure_rankers(self:Vault):
    'Create and migrate the ranker table.'
    t = self.db.t.rankers
    with write_txn(self.db):
        t.create(store=str, model=str, at=float, enabled=int, logging=int, note=str, noise=str,
                 noise_on=int, pk='store', if_not_exists=True)
        have = {c.name for c in t.columns}
        for col, ty in (('noise', 'TEXT'), ('noise_on', 'INTEGER')):
            if col not in have: self.db.conn.execute(f'ALTER TABLE rankers ADD COLUMN {col} {ty}')

@patch
def _ensure_store(self:Vault):
    'Create the shelf registry.'
    with write_txn(self.db):
        self.db.t.vault_stores.create(store=str, encoder=str, dims=int, method=str, added_at=float,
            pk='store', if_not_exists=True)

@patch
def _ensure_marks(self:Vault):
    'Create document marks and its lookup index.'
    with write_txn(self.db):
        self.db.t.doc_marks.create(doc_id=str, store=str, noisy=int, noisy_reason=str,
            pii_override=str, pii_reason=str, at=float, pk=('doc_id', 'store'), if_not_exists=True)
        self.db.conn.execute('CREATE INDEX IF NOT EXISTS doc_marks_noisy ON doc_marks(store, noisy)')

@patch
def _ensure_fb(self:Vault):
    'Create feedback storage and lookup indexes.'
    with write_txn(self.db):
        self.db.t.feedback.create(id=str, at=float, store=str, q=str, ask_id=str, node_id=str,
            doc_id=str, rank=int, score=float, signal=str, label=float, weight=float, pk='id', if_not_exists=True)
        self.db.conn.execute('CREATE INDEX IF NOT EXISTS feedback_doc_id ON feedback(store, doc_id)')
        self.db.conn.execute('CREATE INDEX IF NOT EXISTS feedback_ask_id ON feedback(store, ask_id)')

@patch
def _ensure_schema(self:Vault):
    'Create fixed vault tables once per connection.'
    if getattr(self.db, '_vishalakshi_schema', False): return
    had_marks = 'doc_marks' in self.db.t
    self._ensure_store()
    self._ensure_marks()
    self._ensure_rankers()
    self._ensure_fb()
    if not had_marks: self._migrate_marks()
    self.db._vishalakshi_schema = True

@patch
def _register(self:Vault):
    'Register this shelf and its encoder.'
    self._ensure_schema()
    t, now = self._stores(), time.time()
    r = first(t(where=f'store={self.name!r}'))
    if r and (r['encoder'], r['dims']) != (self.enc.name, self.enc.dims):
        warnings.warn(f"store {self.name!r} was written by {r['encoder']} ({r['dims']}d) but this Vault is "
            f"using {self.enc.name} ({self.enc.dims}d). Distances across the two are meaningless. "
            f"Re-ingest, or keep them apart with shelf('{self.name}-{self.enc.method}').")
    elif not r: t.insert(dict(store=self.name, encoder=self.enc.name, dims=self.enc.dims, method=self.enc.method, added_at=now), replace=True)

@patch
def shelf(self:Vault, name:str, encoder:str=None, **kw) -> Vault:
    'Open another shelf in this vault file.'
    was = first(self._stores()(where=f'store={name!r}')) or {}
    enc = encoder or was.get('encoder')
    if enc is None or enc == self.enc.name: enc = self.enc
    if enc == 'hash': enc, kw = None, dict(kw, offline=True)   # nothing to load; do not try
    return Vault(self.path, encoder=enc, store=name, db=self.db, **kw)

@patch
def drop_shelf(self:Vault, name:str, force:bool=False) -> dict:
    "Delete a shelf and its derived data. Refuse the main shelf unless `force=True`."
    if name == 'store' and not force: raise ValueError("refusing to drop the main shelf; pass force=True")
    pre = '' if name == 'store' else f'{name}_'
    have = {r['name'] for r in self.db.q("select name from sqlite_master where type='table'")}
    gone = []
    for tn in (f'{name}_fts', f'{pre}entities_fts', name, f'{pre}nodes', f'{pre}docs',
               f'{pre}entities', f'{pre}mentions', f'{pre}edges'):
        if tn in have:
            self.db.q(f'DROP TABLE IF EXISTS [{tn}]'); gone.append(tn)
    for r in self.db.q('select path from usearch_indices where name=?', [name]):
        try: Path(r['path']).unlink(missing_ok=True)
        except Exception: pass
    self.db.q('delete from usearch_indices where name=?', [name])
    try: self._stores().delete_where(f'store={name!r}')
    except Exception: pass
    self.db.forget_ensured(name)
    return dict(shelf=name, dropped=gone)

@patch
def shelves(self:Vault) -> L:
    'Shelves with encoder and document count.'
    def n(s):
        p = '' if s == 'store' else f'{s}_'
        try: return self.db.t[f'{p}docs'].count
        except Exception: return 0
    return L(self._stores()(order_by='added_at')).map(lambda r: dict(r, docs=n(r['store'])))

# Shelf names, not encoder assignments
SHELVES = ('store',      # the main shelf: notes, pages, anything unrouted
           'papers',     # arXiv and papers
           'sanskrit',   # veda, commentary, translation: Devanagari and IAST alike
           'code',       # source filed as prose; kosha is the real code index, reached by federate
           'data')       # API harvests and record dumps


## Marks


In [ ]:
#| export
MARK_COLS = ('noisy', 'noisy_reason', 'pii_override', 'pii_reason')

@patch
def _marks(self:Vault):
    'Per-document judgements'
    return self.db.t.doc_marks

@patch
def _noisy_sql(self:Vault) -> str|None:
    """The anti-join `_where` splices in, or None when nothing in this store is marked noisy.
    Cached per Vault, like `_rk`; `mark` clears it."""
    if self._any_noisy is None:
        try: self._any_noisy = bool(first(self._marks()(where=f'store={self.name!r} AND noisy=1', limit=1)))
        except Exception: return None
    if not self._any_noisy: return None
    return f'doc_id NOT IN (SELECT doc_id FROM doc_marks WHERE store={self.name!r} AND noisy=1)'

@patch
def mark(self:Vault, ref, **kv) -> dict:
    "Record a judgement about one document; unknown keys raise."
    if bad := set(kv) - set(MARK_COLS): raise ValueError(f'not a mark: {sorted(bad)}; expected {MARK_COLS}')
    d = self.doc(ref)
    if not d: raise ValueError(f'no document in the vault matching {ref!r}')
    t = self._marks()
    cur = first(t(where=f"doc_id={d['id']!r} AND store={self.name!r}")) or {}
    row = {**{c: None for c in MARK_COLS}, **{k: v for k, v in cur.items() if k in MARK_COLS}, **kv}
    row.update(doc_id=d['id'], store=self.name, at=time.time())
    t.insert(row, replace=True)
    self._any_noisy = None
    return dict(row, title=d['title'])

@patch
def marks(self:Vault, ref=None) -> dict|L:
    "Every judgement on this shelf, or the one on `ref` (an empty dict when there is none)."
    if ref is None: return L(self._marks()(where=f'store={self.name!r}', order_by='at desc'))
    d = self.doc(ref)
    return dict(first(self._marks()(where=f"doc_id={(d or {}).get('id')!r} AND store={self.name!r}")) or {})

@patch
def _migrate_marks(self:Vault) -> int:
    "Copy noisy/pii marks from document `meta` into `doc_marks` (legacy rows)."
    n = 0
    for r in self.t.docs(where="json_extract(meta,'$.noisy') IS NOT NULL "
                               "OR json_extract(meta,'$.pii_override') IS NOT NULL"):
        m = json.loads(r['meta'] or '{}')
        self._marks().insert(dict(doc_id=r['id'], store=self.name, noisy=int(bool(m.get('noisy'))),
            noisy_reason=m.get('noisy_reason') or None, pii_override=m.get('pii_override'),
            pii_reason=m.get('pii_override_reason') or None, at=time.time()), replace=True)
        n += 1
    if n: self._any_noisy = None
    return n


In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)
    assert 'vault_stores' in v.db.t
    assert v._stores().count == 1
    assert v.shelves()[0]['store'] == 'store'

In [ ]:
with TemporaryDirectory() as d:
    p = Path(d)/'test.db'
    a = Vault(p, offline=True)
    a.add('A report about polonium.', title='report')
    b = Vault(p, offline=True)
    assert b.doc('report')
    assert b._stores().count == 1
    assert 'feedback' in b.db.t
    assert 'rankers' in b.db.t

In [ ]:
with TemporaryDirectory() as d:
    p = Path(d)/'test.db'

    a = Vault(p, offline=True)
    a.add('A report about polonium.', title='report')
    before = list(a.marks())

    b = Vault(p, offline=True)
    assert list(b.marks()) == before

In [ ]:
with TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)
    v.add('A report about polonium.', title='report')

    assert 'doc_marks' in v.db.t
    assert v.marks() == []

    v.mark('report', noisy=True)
    assert v.marks('report')['noisy'] == 1
    assert not v.search('polonium')
    assert v.search('polonium', include_noisy=True)

In [ ]:
#| export
@patch
def _post(self:Vault, q:str, ctx):
    """Hook: what `context` hands back, after retrieval and before the federated legs."""
    return ctx

@patch
def _observe(self:Vault, out):
    "Hook: what `ask` hands back. A no-op until `quality.learn()` turns feedback logging on."
    return out


A shelf reuses the vault's live encoder. `$VISHALAKSHI_OFFLINE` is read by `Vault`, not only the CLI.


In [ ]:
# A shelf is a partition, not a second encoder, so it reuses the vault's live one.
from tempfile import mkdtemp

v = Vault(Path(mkdtemp())/'v.db')
s = v.shelf('project')
assert s.enc.model is v.enc.model      # the same object: a shelf per project used to cost a load each
assert s.name == 'project'

# `offline=True` has to survive into a brand-new shelf, or it does not mean "never download":
# a shelf with no registry row used to fall through to the default and load a real model
o = Vault(Path(mkdtemp())/'o.db', offline=True)
assert o.enc.method == 'hash' and o.shelf('fresh').enc.method == 'hash'

# An explicit encoder still wins, and a shelf already on disk is reopened with whatever wrote it.
r = v.shelf('papers', encoder='code')
assert r.enc.name == ENCODERS['code']
r.note('rank fusion needs only each leg ordering', title='RRF')
assert Vault(v.path).shelf('papers').enc.name == ENCODERS['code']
assert Vault(v.path).shelf('papers').search('rank fusion'), 'the reopened shelf lost its note'
print('shelf encoders:', {x['store']: x['encoder'] for x in v.shelves()})

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shelf encoders: {'store': 'minishlab/potion-multilingual-128M', 'project': 'minishlab/potion-multilingual-128M', 'papers': 'minishlab/potion-retrieval-32M'}


`KIND_SHELF` is where acquisition routes, deliberately one entry. `find` and `sections` are single-shelf; pass `shelves=` or use `elsewhere` when the answer may live next door.


In [ ]:
#|export
KIND_SHELF = {'arxiv': 'papers', 'sanskrit': 'sanskrit'}

def is_sanskrit_file(path) -> bool:
    'Whether a Sanskrit reader is registered for this file. False until ganapati is imported.'
    from litesearch.data import profile_for
    p = Path(path)
    return p.is_file() and (pr := profile_for(p)) is not None and (pr.kind or '') == 'sanskrit'

_facets_on = False
def sanskrit_facets() -> bool:
    '''Re-register Sanskrit profiles with lemmas and Monier-Williams glosses in metadata.'''
    global _facets_on
    if _facets_on: return True
    try:
        from ganapati import register_profiles, vidyut_pipe, mw_lexicon
        register_profiles(nlp=vidyut_pipe(), mw=mw_lexicon())
        _facets_on = True
    except Exception as e:
        warnings.warn(f'Sanskrit lemmas and glosses unavailable ({type(e).__name__}: {str(e)[:80]}); '
                      'indexing metre only. Retrieval by English paraphrase will be weaker.')
    return _facets_on

@patch
def route(self:Vault, kind:str) -> Vault:
    'The shelf a `kind` belongs on: `self`, unless `KIND_SHELF` sends it elsewhere.'
    nm = KIND_SHELF.get(kind)
    if nm == 'sanskrit': sanskrit_facets()   # before the ingest that is about to happen, not after
    return self.shelf(nm) if nm and self.name == 'store' and nm != self.name else self

@patch
def elsewhere(self:Vault,
              q:str,              # the question
              limit:int=2,        # sections taken from each other shelf
              shelves=True,       # True -> every other shelf; a list picks some
              max_read:int=2000,  # chars kept per section
) -> L:
    "Sections from the vault's *other* shelves, shaped like this one's."
    out, want = L(), (None if shelves is True else set(L(shelves)))
    for s in self.shelves():
        nm = s['store']
        if nm == self.name or not s['docs'] or (want is not None and nm not in want): continue
        for r in self.shelf(nm).sections(q, limit=limit): out.append(AttrDict(node_id=r['node_id'],
            doc_id=r['node_id'].split('#')[0], store=nm, title=r['title'], breadcrumb=f"{nm} › {r['breadcrumb']}",
            filename=None, pages=r['pages'], via=f'shelf:{nm}', text=' '.join(r['snippets'])[:max_read]))
    return out


In [ ]:
with TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)
    v.add('A report about polonium.', title='report')

    sql = []
    v.db.conn.set_exec_trace(lambda cursor, statement, bindings:sql.append(statement) or True)

    v.search('polonium')
    v.sections('polonium')
    v.context('polonium', code=0, shelves=0)

    v.db.conn.set_exec_trace(None)

    ddl = [
        q for q in sql
        if q.lstrip().upper().startswith(('CREATE ', 'ALTER ', 'DROP '))
    ]
    assert ddl == [], ddl

In [ ]:
#| export
def _paged(tbl, batch:int=2000):
    'Rows out of `tbl` a page at a time, so nothing has to hold the whole corpus at once.'
    if not batch: yield from tbl(); return   # `batch=0` is "do not page", not "page by nothing"
    off = 0
    while (rows := list(tbl(limit=batch, offset=off, order_by='rowid'))):
        yield from rows
        off += len(rows)

@patch
def connect(self:Vault,
            resolve:bool=True,   # merge duplicate entities once the build is in
            topics:bool=True,    # (re)write the labelled topic nodes
            batch:int=2000,      # chunks per flush; keeps co-occurrence windows on disk, not in memory
            n_workers:int=None,  # extraction workers; 0 is serial, None picks by queue size
            **kw) -> dict:
    '(Re)build the entity graph over everything in the vault.'
    if not self.store.count: return dict(entities=0, mentions=0, edges=0, windows=0)
    if 'terms_fn' not in kw:
        try:
            from ganapati import is_sanskrit, sanskrit_terms
            head = ' '.join(c['content'] or '' for c in self.store(limit=20))
            if is_sanskrit(head): kw['terms_fn'] = sanskrit_terms()
        except Exception: pass          # no ganapati, so fall back to whatever the extractor defaults to
    from vruksha import build_graph, resolve_entities
    self.db.get_graph(self.name, ndim=self.enc.dims, dtype=DTYPE)
    res = build_graph(self.db, _paged(self.store, batch), store=self.name, emb_fn=self.emb,
                      batch=batch, n_workers=n_workers, **kw)
    if resolve: res = dict(res, resolved=resolve_entities(self.db, store=self.name, dtype=DTYPE))
    if topics:
        g = self.db.get_graph(self.name)
        try: g.mentions.delete_where(f"entity_id IN (SELECT id FROM {g.prefix}entities WHERE kind='topic')")
        except Exception: pass
        try: g.entities.delete_where("kind='topic'")
        except Exception: pass
        res = dict(res, **topic_nodes(self.db, store=self.name, dtype=DTYPE))
    return res

def _map_from_graph(db, store, store_table, members:int=24) -> AttrDict|None:
    'Read topic clusters persisted by `connect()`: instant DB read, no re-clustering.'
    try: g = db.get_graph(store)
    except Exception: return None
    try: ents = L(g.entities(where="kind='topic'", order_by='freq desc'))
    except Exception: return None
    if not ents: return None
    eids = ','.join(repr(e['id']) for e in ents)
    cid_topic = {}
    for m in g.mentions(select='chunk_id, entity_id', where=f'entity_id IN ({eids})'):
        cid_topic.setdefault(m['entity_id'], []).append(m['chunk_id'])
    want = list({c for cs in cid_topic.values() for c in cs[:members]})
    rows = ({r['id']: r for r in store_table(select='id, content, doc_id',
             where=f"id IN ({','.join(repr(i) for i in want)})")} if want else {})
    clusters = L(AttrDict(centroid=None, size=e['freq'],
                          label=e['content'].removeprefix('topic: '),
                          member_keys=cid_topic.get(e['id'], []),
                          members=L(rows[c] for c in cid_topic.get(e['id'], [])[:members] if c in rows))
                 for e in ents)
    return AttrDict(clusters=clusters, method='cached', note=f'{len(clusters)} cached topics from graph')

@patch
def map(self:Vault, min_count:int=2, force:bool=False, **kw) -> AttrDict:
    'Cluster the corpus into labelled topics: the shape of what you have collected. Fast after `connect()` has run: reads the persisted topic nodes rather than re-clustering.'
    if not force:
        cached = _map_from_graph(self.db, self.name, self.store)
        if cached is not None: return cached
    return self.store.clusters(min_count=min_count, dtype=DTYPE, columns=['content', 'doc_id'], **kw)

def _sql_in(col, xs, batch:int=2000):
    "`col IN (...)` clauses over `xs`, split so no one statement grows unbounded."
    xs = list(xs)
    for i in range(0, len(xs), batch):
        yield f"{col} IN ({','.join(repr(x) for x in xs[i:i+batch])})"

@patch
def topic_tree(self:Vault,
               limit:int=20,      # topics returned, largest first
               docs:int=8,        # documents listed under each topic
               min_chunks:int=2,  # skip a topic carried by fewer chunks than this
) -> L:
    '''Topics, and which documents each one runs through. The shape of the corpus, two levels deep.'''
    try: g = self.db.get_graph(self.name)
    except Exception: return L()
    try: ents = L(g.entities(where="kind='topic'", order_by='freq desc'))
    except Exception: return L()
    if not ents: return L()
    by_topic = {}
    for w in _sql_in('entity_id', [e['id'] for e in ents]):
        for m in g.mentions(select='chunk_id, entity_id', where=w):
            by_topic.setdefault(m['entity_id'], []).append(m['chunk_id'])
    cid_doc = {}
    for w in _sql_in('id', {c for cs in by_topic.values() for c in cs}):
        for r in self.store(select='id, doc_id', where=w): cid_doc[r['id']] = r['doc_id']
    # same title can appear in many repos; keep both under one topic
    drows = {r['id']: r for r in self.t.docs(select='id, title, source')}
    dupes = Counter(r['title'] for r in drows.values())
    def _name(d):
        r = drows.get(d)
        if not r: return d
        t = r['title'] or d
        if dupes[t] <= 1 or not r['source']: return t
        parts = Path(r['source']).parts          # the parent is what tells two `index.ipynb`s apart
        return f"{t} ({'/'.join(parts[-2:])})" if len(parts) > 1 else f"{t} ({d[:8]})"
    out = L()
    for e in ents:
        cs = by_topic.get(e['id'], [])
        if len(cs) < min_chunks: continue
        n = Counter(d for c in cs if (d := cid_doc.get(c)))
        out.append(AttrDict(label=(e['content'] or '').removeprefix('topic: '), topic_id=e['id'],
                            size=e['freq'], chunks=len(cs), docs=len(n),
                            sources=L(AttrDict(doc_id=d, title=_name(d), chunks=k)
                                      for d, k in n.most_common(docs))))
        if len(out) >= limit: break
    return out

def fmt_topics(tree, width:int=44) -> str:
    "A `topic_tree` as an indented listing. Plain ASCII, so it survives a terminal, a notebook and a prompt."
    if not tree: return 'no topics: run connect() first'
    lines = []
    for t in tree:
        lines.append(f"{t['label'][:width]:<{width}} ({t['chunks']} chunks, {t['docs']} docs)")
        srcs = t['sources']
        for i, s in enumerate(srcs):
            stem = '`- ' if i == len(srcs)-1 else '|- '
            lines.append(f"  {stem}{str(s['title'])[:width-4]:<{width-4}} {s['chunks']:>4}")
    return '\n'.join(lines)

@patch
def show_topics(self:Vault, limit:int=20, docs:int=8, **kw):
    "Print `topic_tree` as an indented listing."
    print(fmt_topics(self.topic_tree(limit=limit, docs=docs, **kw)))

@patch
def sources(self:Vault, kind:str=None, include_noisy:bool=True) -> L:
    'Documents with provenance, newest first; management sees noisy rows unless asked not to.'
    wh = ' AND '.join(x for x in (_kw(kind) if kinds(kind) else '',
                                  f'id NOT IN (SELECT doc_id FROM doc_marks WHERE store={self.name!r} AND noisy=1)'
                                  if not include_noisy else '') if x) or None
    rows = self.t.docs(where=wh, order_by='added_at desc')
    mk = {r['doc_id']: r for r in self._marks()(where=f'store={self.name!r}')}
    return L(rows).map(lambda d: dict(d, meta=json.loads(d['meta'] or '{}'),
                                      noisy=bool((mk.get(d['id']) or {}).get('noisy'))))

@patch
def mark_noisy(self:Vault, ref, noisy:bool=True, reason:str='') -> dict:
    """Mark a document as retrieval noise: search, sections, context and ask exclude it by default."""
    return self.mark(ref, noisy=int(bool(noisy)), noisy_reason=(reason or None) if noisy else None)

@patch
def forget(self:Vault, doc_id:str):
    'Remove a document, its sections and its chunks, and rebuild the ANN index.'
    self.db.delete_doc(doc_id, store=self.name)

@patch
def stats(self:Vault) -> dict:
    'Row counts across the vault, by kind.'
    p, t = self.t.prefix, self.db.t
    ents = (first(self.db.q(f"SELECT COUNT(*) n FROM {p}entities WHERE kind!='topic'")) or {}).get('n', 0) if f'{p}entities' in t else 0
    return dict(docs=self.t.docs.count, nodes=self.t.nodes.count, chunks=self.store.count, encoder=self.enc.method,
                entities=ents, path=self.path,
                by_kind={r['kind']: r['n'] for r in self.db.q(f'select kind, count(*) as n from {p}docs group by kind order by n desc')})


In [ ]:
with TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)
    v.add('Main report about polonium.', title='main')

    papers = v.shelf('papers')
    papers.add('Paper about thorium.', title='paper')

    sql = []
    v.db.conn.set_exec_trace(
        lambda cursor, statement, bindings:
            sql.append(statement) or True
    )

    v.context('thorium', code=0, shelves=2)

    v.db.conn.set_exec_trace(None)

    ddl = [
        q for q in sql
        if q.lstrip().upper().startswith(('CREATE ', 'ALTER ', 'DROP '))
    ]
    assert ddl == [], ddl

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    v = Vault(Path(d)/'test.db', offline=True)
    v.add("A scheduled report about polonium.", title="report")
    result = v.connect(n_workers=0)

    assert result['entities'] > 0
    assert v.search('polonium')
    result

## Try it

`offline=True` uses litesearch's `hash_embed` (lexical retrieval; `stats()['encoder']` says so).


In [ ]:
v = Vault(':memory:')
v.note('Late chunking beats naive chunking because context survives the split.', tags=['retrieval'])
v.add('# Attention\n\nScaled dot-product attention weights values by query-key similarity.\n\n'
      '## Multi-head\n\nHeads attend to different subspaces in parallel.', 'Attention', kind='note')
v.stats()

{'docs': 2,
 'nodes': 5,
 'chunks': 3,
 'encoder': 'model2vec',
 'entities': 0,
 'path': ':memory:',
 'by_kind': {'note': 2}}

In [ ]:
test_eq(v.stats()['docs'], 2)
test_eq(v.stats()['encoder'], 'model2vec')
assert v.search('chunking')[0]['snippet']
# a hit is a handle plus enough text to recognise it, not the chunk itself; the scoring
# internals the two legs disagree about (`rank`, `_dist`, `_rrf_score`, `rowid`) stay inside
_h = v.search('chunking')[0]
test_eq(sorted(_h), ['breadcrumb', 'doc_id', 'node_id', 'page', 'score', 'snippet'])
assert len(v.search('chunking', chars=20)[0]['snippet']) <= 20
assert v.read(_h['node_id'])['text']          # the handle opens what the snippet came from
test_eq([d['kind'] for d in v.sources()], ['note', 'note'])
test_eq(len(v.sources(kind='web')), 0)
test_eq(len(v.search('chunking', kind='web')), 0)

In [ ]:
d = v.doc('Attention')                          # by title substring
test_eq(v.doc(d['id'])['title'], 'Attention')   # by doc_id
test_eq(v.doc(d['source'])['title'], 'Attention')  # by source
test_eq(v.doc('nothing in here'), None)

whole = v.document('Attention')
test_eq((whole.origin, whole.nodes), ('vault', 3))   # the root, the heading, the subheading
# every section, in document order, with the headings that say what each one is
assert whole.text.index('Scaled dot-product') < whole.text.index('## Multi-head') < whole.text.index('subspaces')
test_eq(v.document('Attention', headings=False).text.find('## Multi-head'), -1)
test_eq(v.document('Attention', max_chars=20).text, whole.text[:20])
test_eq(v.document('Attention', max_chars=20).truncated, True)
# a `Pages n–m:` node title is build_tree's placeholder, not a heading the document wrote
v.add('Prose with no headings at all, long enough to chunk and to be stored in the vault.', 'plain')
assert '# Pages' not in v.document('plain').text and 'Prose with no' in v.document('plain').text

test_eq(v.set_meta(d['id'], doctype='paper')['doctype'], 'paper')
test_eq(v.doc(d['id'])['meta']['doctype'], 'paper')                 # survives the round trip
test_eq(v.set_meta(d['id'], reviewed=True)['doctype'], 'paper')     # merged, not replaced
test_fail(lambda: v.document('no such document'), contains='no document in the vault')

In [ ]:
#| hide
# Three aliases, each a plain string, so reading the table costs no import and no download.
test_eq(set(ENCODERS), {'multilingual', 'code', 'gemma'})
assert all(isinstance(v, str) for v in ENCODERS.values())
test_eq(ENCODERS['multilingual'], DFLT_ENC)

# `gemma` is an ONNX id, and naming it must not pull onnxruntime in: `_load` imports FastEncode
# only when gemma is the encoder actually being built.
import sys
assert 'onnxruntime' not in sys.modules

class _E:            # an embedder you built yourself: litesearch's doc_encoder takes anything with .encode
    def encode(self, xs, **kw): return np.zeros((len(xs), 8), dtype=np.float16)
test_eq((mk_encoder(_E()).dims, mk_encoder(_E()).method), (8, 'given'))
test_eq(mk_encoder('no/such-model-at-all', offline=True).method, 'hash')

# the vault's default and a bare litesearch Index have to agree, or one file holds two vector
# spaces under one name and nothing says so
test_eq(DFLT_ENC, 'minishlab/potion-multilingual-128M')

In [ ]:
#| hide
# every alias is a real hub id rather than a local path
for _a, _m in ENCODERS.items(): assert '/' in _m and not Path(_m).exists(), _a
# SHELVES is a tuple of names, not an encoder map: one encoder writes every shelf
assert isinstance(SHELVES, tuple) and 'sanskrit' in SHELVES and 'papers' in SHELVES
assert all(isinstance(s, str) for s in SHELVES)

In [ ]:
#| hide
# a model that will not load must degrade to the hash encoder rather than leaving a broken vault
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    e = mk_encoder('no/such-model-at-all')
test_eq(e.method, 'hash')
test_eq(e.model.encode(['probe']).shape, (1, e.dims))
test_eq(e.model.encode(['probe']).dtype, np.float16)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _bad = Vault(':memory:', encoder='no/such-model-at-all')
test_eq(_bad.enc.method, 'hash')                     # and the vault opens
test_eq(_bad.emb(['probe']).shape, (1, e.dims))      # embedding through Index still works

In [ ]:
#| hide
# a shelf is a partition of one file: its own store and tree, one shared connection
sh = v.shelf('papers')
sh.add('Contextual chunk embeddings keep the document around the chunk.', 'a paper')
test_eq((sh.name, sh.db is v.db), ('papers', True))     # shared, or ':memory:' would be a second db
test_eq([s['store'] for s in v.shelves()], ['store', 'papers'])
test_eq(v.doc('a paper'), None)                         # a partition, not a second index over the same docs
test_eq(sh.doc('a paper')['title'], 'a paper')
test_eq(v.shelf('papers').enc.method, 'model2vec')      # reopened with the encoder that wrote it
test_eq(v.shelf('sanskrit').name, 'sanskrit')
# every shelf is one vector space wide, which is what lets a shelf be a plain Index
test_eq({s['dims'] for s in v.shelves()}, {v.enc.dims})
test_eq((v.route('arxiv').name, v.route('web').name), ('papers', 'store'))

# reopening a store with a *different* encoder is silent and total, so it has to be said out loud
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    v.shelf('papers', encoder=_E())
    assert any('meaningless' in str(x.message) for x in w), [str(x.message) for x in w]

# ...and what makes a library of shelves usable is reading across it
e = v.elsewhere('chunk embeddings')
test_eq(e.attrgot('store'), ['papers'])
assert e[0].breadcrumb.startswith('papers › ') and e[0].text
assert v.read(e[0].node_id, store='papers')['text']   # a citation into a shelf has to open
test_eq(v.read(e[0].node_id), {})                     # and the wrong store finds nothing
test_eq(v.elsewhere('chunk embeddings', shelves=['nowhere']), [])
# the sanskrit shelf above is registered but empty, and an empty shelf is skipped rather than opened
assert 'sanskrit' in v.shelves().attrgot('store')
test_eq(sh.elsewhere('late chunking', limit=1).attrgot('store'), ['store'])   # reads both ways
test_eq(len(sh.elsewhere('late chunking', limit=2)), 2)                      # limit is per shelf

c = v.context('chunk embeddings', sections=2, related=0, code=0)
test_eq(c.shelves, 1)
assert any(r.get('store') == 'papers' for r in c.results), c.results
test_eq(v.context('chunk embeddings', related=0, code=0, shelves=0).shelves, 0)

# a shelf's encoder cannot be migrated in place, so dropping is the supported way to change it
_dropped = v.drop_shelf('papers')
assert 'papers' in _dropped['dropped'] and 'papers_docs' in _dropped['dropped'], _dropped
test_eq([s['store'] for s in v.shelves()], ['store', 'sanskrit'])   # gone from the registry too
test_eq(v.elsewhere('chunk embeddings'), [])                        # and from every read across
test_fail(lambda: v.drop_shelf('store'), contains='refusing to drop')
sh = v.shelf('papers')                                              # rebuilt empty, and reusable
sh.add('Contextual chunk embeddings keep the document around the chunk.', 'a paper')
test_eq(sh.doc('a paper')['title'], 'a paper')

In [ ]:
#| hide
from tempfile import mkdtemp
td = Path(mkdtemp()).resolve()          # litesearch reports the resolved path; /var is a symlink
vf = Vault(str(td/'v.db'), offline=True)
test_eq(vf.assets(), td/'assets')
test_eq(vf.assets('paper'), td/'assets'/'paper')

p = td/'fusion_notes.md'
p.write_text('# Fusion\n\nRanks are fused because the legs share no vector space.\n')
r = vf.add_file(p)
test_eq(r['kind'], 'md')                              # litesearch reads it off the extension
test_eq(r['title'], 'fusion notes')                   # and prettifies the filename
assert vf.search('rank fusion')

r = vf.add_file(p, title='Fusion, filed', kind='note')
test_eq(r['kind'], 'note')                            # an explicit kind wins, at ingest not after
test_eq(first(vf.t.docs(where=f'id={r["doc_id"]!r}'))['kind'], 'note')   # and it is what landed

# a shelf shares the vault's database, so it shares the vault's asset directory
test_eq(vf.shelf('papers', offline=True).assets(), vf.assets())
# an in-memory vault has no directory of its own; assets must not fall back to the cwd
assert not str(Vault(':memory:', offline=True).assets()).startswith(str(Path.cwd()))

# `embed_batch` decides *when* chunks are written, not which ones: batched across documents
# and written one document at a time have to land the same corpus, or the fast path is a
dd = td/'batched'; dd.mkdir()
for _i in range(6): (dd/f'n{_i}.md').write_text(f'# Note {_i}\n\nReciprocal rank fusion merges two ranked lists.\n')
_fs = sorted(dd.glob('*.md'))
vb = Vault(str(td/'b.db'), offline=True); vb.add_files(_fs, embed_batch=2)
vs = Vault(str(td/'s.db'), offline=True); vs.add_files(_fs, embed_batch=0, n_workers=0)
test_eq(vb.store.count, vs.store.count)
test_eq(vb.stats()['docs'], 6)
assert vb.search('rank fusion')        # and the ANN index was rebuilt, once, after the last flush
test_eq(len(vb.add_files(_fs)), 6)   # content-addressed, so a second pass re-reports rather than duplicates
test_eq(vb.stats()['docs'], 6)

# the parse pool is the other branch, and `n_workers` is the only way to reach it from a
# corpus of markdown: `None` counts parse-heavy extensions and picks serial for these
vp = Vault(str(td/'p.db'), offline=True); _op = vp.add_files(_fs, n_workers=4)
test_eq((vp.stats()['docs'], vp.store.count), (6, vs.store.count))
test_eq(sum(o is None for o in _op), 0)      # every file came back, parsed in a worker or here
assert vp.search('rank fusion')
test_eq(vb.add_files([]), L())               # and no files does no work, not an index rebuild

# batch only changes where co-occurrence windows live; the graph must match
def _graph_of(batch):
    _vg = Vault(str(td/f'g{batch}.db'), offline=True)
    _vg.add_files(_fs); _vg.connect(batch=batch)
    _g = _vg.db.get_graph(_vg.name)
    return dict(entities=_g.entities.count, mentions=_g.mentions.count, edges=_g.edges.count,
                n=sum(m['n'] for m in _g.mentions(select='n')))
test_eq(_graph_of(0), _graph_of(1))     # unbatched, and a flush per chunk
test_eq(_graph_of(0), _graph_of(4))


In [ ]:
#| hide
# an nbdev project keeps a generated copy of every notebook in `_proc` and a rendered one in
# `_docs`
_nb = td/'proj'; (_nb/'nbs').mkdir(parents=True); (_nb/'_proc').mkdir(); (_nb/'_docs').mkdir()
_body = '# Fusion\n\nRanks are fused because the legs share no vector space.\n'
for _sub in ('nbs', '_proc', '_docs'): (_nb/_sub/'index.md').write_text(_body)
_vs = Vault(str(td/'skip.db'), offline=True)
_vs.add(str(_nb))
test_eq(_vs.stats()['docs'], 1)
test_eq([Path(d['source']).parent.name for d in _vs.sources()], ['nbs'])
# skipped by name, not by having been seen before
test_eq(len(dir2files(_nb)), 1)
test_eq(dir2files(_nb)[0].parent.name, 'nbs')

In [ ]:
test_eq(tidy_bc('Attention › Pages 1–1: Scaled dot-product › Multi-head'), 'Attention › Multi-head')
test_eq(tidy_bc(None), '')
# a window with a blank first line leaves the placeholder bare, and it is still a placeholder
test_eq(tidy_bc('Doc › Pages 1–1: › Body'), 'Doc › Body')

In [ ]:
r = v.connect()
assert r['entities'] > 0 and r['resolved']['resolvable'] == r['entities']
test_eq(v.stats()['entities'], r['entities'])

In [ ]:
#| hide
# topic_tree is the other half of map(): map says what the subjects are, this says where each lives
_tt = v.topic_tree(limit=3, docs=2)
assert _tt, 'connect() ran, so there are topic nodes to read'
test_eq(sorted(_tt[0]), ['chunks', 'docs', 'label', 'size', 'sources', 'topic_id'])
assert all(t['chunks'] >= 2 for t in _tt)                    # min_chunks is honoured
assert all(len(t['sources']) <= 2 for t in _tt)              # and so is the per-topic doc cap
assert all(s['title'] for t in _tt for s in t['sources'])    # every source resolves to a real title
test_eq(len(v.topic_tree(limit=1)), 1)
assert 'topic: ' not in _tt[0]['label']                      # the storage prefix is not the label

# the renderer is plain ascii, and says so rather than raising when there is nothing to show
_s = fmt_topics(_tt)
assert _tt[0]['label'][:20] in _s and _s.isascii(), _s
test_eq(fmt_topics([]), 'no topics: run connect() first')
# a vault with no graph at all must answer, not explode
test_eq(Vault(':memory:', offline=True).topic_tree(), [])

In [ ]:
#| hide
test_eq(v.enc.dims, 256)
test_eq(len(v.qemb('chunking')), v.enc.dims*2)      # float16 bytes, matching what the store holds
test_eq(v.emb(['a', 'b']).shape, (2, v.enc.dims))
test_eq(v.emb(['a']).dtype, DTYPE)
with warnings.catch_warnings():
    warnings.simplefilter('error')          # litesearch warns on a dtype mismatch; it must not fire
    assert v.context('why does late chunking help', sections=2, related=2,
                     code=0, shelves=0).results   # this cell is about *this* store's width

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()